## tl;dr

- Target **65% gross margin** on implementations, with a **60% hard floor**.
- Target **70–75%** on support and repeatable diagnostics.
- Recommended standard ex-GST prices are $3.2k Heutrix Diagnostics, $16k standard Workflow Transformation, $19.5k visibility-focused Workflow Transformation, $9.5k Heutrix AI Guardrails, $33k expanded Workflow Transformation and $1.8k/month support.
- At the original Base volumes this plan produces about $287k, $515k and $761k revenue with roughly 66–67% blended gross margin at a $100 loaded delivery cost per hour.

## Context & Methods

This notebook reconciles the product prices, modeled loaded delivery costs and Base-case unit volumes in the existing Heutrix three-year forecast. All values are AUD excluding GST.

### Key Assumptions

- Forecast years are the first three 12-month periods after commercial readiness.
- Loaded delivery cost includes direct delivery labour, contractors, attributable tools, project management, training, documentation, stabilisation and rework contingency.
- Delivery is tested at $80, $100 and $140 per loaded hour, representing founder/hybrid, scale-safe mixed and senior-specialist delivery.
- Fixed operating expenses, tax and working capital are outside gross margin.
- Market evidence is supplier-published context reviewed on 1 September 2026, not historical Heutrix cost or validated willingness to pay.
- Source model: `Heutrix-Labs-3-Year-Growth-Forecast.xlsx`, Assumptions and Forecast sheets.

## Data

### 1. Load the derived price plan

In [1]:
from pathlib import Path
import json

notebook_dir = Path.cwd()
model_path = notebook_dir / 'Heutrix-Pricing-and-Margin-Plan.json'
model = json.loads(model_path.read_text(encoding='utf-8'))
pricing = model['pricingPlan']
annual = model['annualPlan']
annual_rates = model['annualRateScenarios']
sensitivity = model['rateSensitivity']
ceilings = model['ceilingInclusions']
len(pricing), len(annual), len(annual_rates), len(sensitivity), len(ceilings)

(6, 3, 9, 18, 6)

### 2. Verify the core formulas and forecast tie-outs

In [2]:
def close(a, b, tolerance=0.01):
    return abs(a - b) <= tolerance

for row in pricing:
    assert close(row['modeled_margin'], 1 - row['loaded_cost'] / row['forecast_price'], 0.0001)
    if row['current_website_price'] > 0:
        assert close(row['current_website_margin'], 1 - row['loaded_cost'] / row['current_website_price'], 0.0001)
    else:
        assert row['current_website_margin'] is None
    assert close(row['recommended_margin'], 1 - row['loaded_cost'] / row['recommended_price'], 0.0001)
    assert close(row['max_cost_at_target'], row['recommended_price'] * (1 - row['target_margin']))
    assert close(row['required_price_at_target'], row['loaded_cost'] / (1 - row['target_margin']))

expected_revenue = [182000, 328000, 486000]
expected_cost = [69500, 123000, 180600]
for index, row in enumerate(annual):
    assert row['forecast_revenue'] == expected_revenue[index]
    assert row['forecast_cogs'] == expected_cost[index]
    assert close(row['forecast_gross_margin'], 1 - expected_cost[index] / expected_revenue[index], 0.0001)
    assert close(row['cost_ceiling_at_65_margin'], expected_revenue[index] * 0.35)
    assert row['scale_safe_revenue'] in [286800, 515000, 760500]
    assert row['scale_safe_cogs'] in [96700, 170900, 250100]
assert {row['rate'] for row in sensitivity} == {80, 100, 140}
assert len(annual_rates) == 9
'All pricing formulas, hourly-rate cases and annual forecast tie-outs passed.'

'All pricing formulas, hourly-rate cases and annual forecast tie-outs passed.'

## Results

### 3. Offer-level economics

In [3]:
headers = ['Offer', 'Website price', 'Website margin', 'Launch floor', 'Standard price', 'Ceiling', 'Loaded cost at $100/h', 'Target margin', 'Maximum cost at standard price', 'Standard margin']
print(' | '.join(headers))
print(' | '.join(['---'] * len(headers)))
for row in pricing:
    website_price = 'Not published' if row['current_website_price'] == 0 else f"${row['current_website_price']:,.0f}"
    website_margin = 'Not published' if row['current_website_margin'] is None else f"{row['current_website_margin']:.1%}"
    print(' | '.join([
        row['offer'],
        website_price,
        website_margin,
        f"${row['launch_price']:,.0f}",
        f"${row['recommended_price']:,.0f}",
        f"${row['ceiling_price']:,.0f}",
        f"${row['loaded_cost']:,.0f}",
        f"{row['target_margin']:.0%}",
        f"${row['max_cost_at_target']:,.0f}",
        f"{row['recommended_margin']:.1%}",
    ]))

Offer | Website price | Website margin | Launch floor | Standard price | Ceiling | Loaded cost at $100/h | Target margin | Maximum cost at standard price | Standard margin
--- | --- | --- | --- | --- | --- | --- | --- | --- | ---
Heutrix Diagnostics | $950 | -15.8% | $2,600 | $3,200 | $4,500 | $1,100 | 65% | $1,120 | 65.6%
Heutrix Workflow Transformation | $2,500 | -124.0% | $13,500 | $16,000 | $22,000 | $5,600 | 65% | $5,600 | 65.0%
Heutrix Workflow Transformation — Visibility Scope | $3,500 | -94.3% | $16,000 | $19,500 | $26,500 | $6,800 | 65% | $6,825 | 65.1%
Heutrix AI Guardrails | $1,800 | -83.3% | $8,000 | $9,500 | $13,000 | $3,300 | 65% | $3,325 | 65.3%
Heutrix Workflow Transformation — Expanded Scope | $4,500 | -155.6% | $27,500 | $33,000 | $45,000 | $11,500 | 65% | $11,550 | 65.1%
Stabilisation / support plan | Not published | Not published | $1,500 | $1,800 | $2,500 | $450 | 75% | $450 | 75.0%


### 4. Loaded-rate sensitivity

In [4]:
headers = ['Loaded rate', 'Offer', 'Loaded cost', 'Required price', 'Package ceiling', 'Within ceiling']
print(' | '.join(headers))
print(' | '.join(['---'] * len(headers)))
for row in sensitivity:
    print(' | '.join([
        f"${row['rate']:,.0f}/h",
        row['offer'],
        f"${row['loaded_cost']:,.0f}",
        f"${row['required_price']:,.0f}",
        f"${row['package_ceiling']:,.0f}",
        row['within_ceiling'],
    ]))

Loaded rate | Offer | Loaded cost | Required price | Package ceiling | Within ceiling
--- | --- | --- | --- | --- | ---
$80/h | Heutrix Diagnostics | $900 | $2,571 | $4,500 | Yes
$80/h | Heutrix Workflow Transformation | $4,600 | $13,143 | $22,000 | Yes
$80/h | Heutrix Workflow Transformation — Visibility Scope | $5,600 | $16,000 | $26,500 | Yes
$80/h | Heutrix AI Guardrails | $2,700 | $7,714 | $13,000 | Yes
$80/h | Heutrix Workflow Transformation — Expanded Scope | $9,500 | $27,143 | $45,000 | Yes
$80/h | Stabilisation / support plan | $370 | $1,480 | $2,500 | Yes
$100/h | Heutrix Diagnostics | $1,100 | $3,143 | $4,500 | Yes
$100/h | Heutrix Workflow Transformation | $5,600 | $16,000 | $22,000 | Yes
$100/h | Heutrix Workflow Transformation — Visibility Scope | $6,800 | $19,429 | $26,500 | Yes
$100/h | Heutrix AI Guardrails | $3,300 | $9,429 | $13,000 | Yes
$100/h | Heutrix Workflow Transformation — Expanded Scope | $11,500 | $32,857 | $45,000 | Yes
$100/h | Stabilisation / support pla

### 5. Annual margin reconciliation at the matching price tier

In [5]:
headers = ['Rate', 'Year', 'Price tier', 'Revenue', 'Direct cost', 'Gross margin', 'Fixed opex', 'Operating contribution', 'Contribution margin']
print(' | '.join(headers))
print(' | '.join(['---'] * len(headers)))
for row in annual_rates:
    print(' | '.join([
        f"${row['rate']:,.0f}/h",
        row['year'],
        row['recommended_price_tier'],
        f"${row['selected_revenue']:,.0f}",
        f"${row['direct_cost']:,.0f}",
        f"{row['selected_gross_margin']:.1%}",
        f"${row['fixed_operating_expense']:,.0f}",
        f"${row['operating_contribution']:,.0f}",
        f"{row['operating_contribution_margin']:.1%}",
    ]))

Rate | Year | Price tier | Revenue | Direct cost | Gross margin | Fixed opex | Operating contribution | Contribution margin
--- | --- | --- | --- | --- | --- | --- | --- | ---
$80/h | Year 1 | Launch floor | $238,700 | $79,460 | 66.7% | $55,000 | $104,240 | 43.7%
$80/h | Year 2 | Launch floor | $428,800 | $140,460 | 67.2% | $85,000 | $203,340 | 47.4%
$80/h | Year 3 | Launch floor | $633,400 | $205,580 | 67.5% | $120,000 | $307,820 | 48.6%
$100/h | Year 1 | Standard list | $286,800 | $96,700 | 66.3% | $55,000 | $135,100 | 47.1%
$100/h | Year 2 | Standard list | $515,000 | $170,900 | 66.8% | $85,000 | $259,100 | 50.3%
$100/h | Year 3 | Standard list | $760,500 | $250,100 | 67.1% | $120,000 | $390,400 | 51.3%
$140/h | Year 1 | Package ceiling | $394,500 | $131,180 | 66.8% | $55,000 | $208,320 | 52.8%
$140/h | Year 2 | Package ceiling | $708,500 | $231,780 | 67.3% | $85,000 | $391,720 | 55.3%
$140/h | Year 3 | Package ceiling | $1,046,500 | $339,140 | 67.6% | $120,000 | $587,360 | 56.1%


## Takeaways

1. The current $950–$2,500 starting prices are below a scale-safe level for the modeled full scopes; no modeled implementation row clears the 60% floor at any tested rate.
2. The standard list creates about 58% revenue headroom over the original forecast and a scale-safe 66–67% margin at $100/hour without changing Base sales volumes.
3. The launch floor is required around $80 per loaded hour, the standard list near $100, and delivery near $140 requires the package ceilings.
4. Any offer below 60% actual gross margin should be re-scoped or repriced before the next sale, and work beyond a package ceiling should be phased.